# Pilot runs — slm-audio-evidence (Kaggle)

Notebook **Settings** (правая панель) → **Accelerator: GPU T4 x2** (НЕ P100 -- Pascal, compute capability 6.0, несовместим с preinstalled PyTorch в образе Kaggle; ячейка ниже проверяет это автоматически и падает с понятной ошибкой, если аккселератор не тот) → **Internet: On** (нужен для `git clone`/`pip install`/скачивания весов). Затем — ячейки сверху вниз.

**Все настраиваемые параметры собраны в ОДНОЙ ячейке ниже (`## Parameters`)** — меняешь значения там, дальше просто выполняешь ячейки по порядку (или Run All целиком). Ни одну ячейку ниже не нужно вручную комментить/раскомментить, чтобы что-то не сломать или не запустить лишний раз — каждая опциональная/дорогая операция управляется своим флагом из Parameters, а не ручной правкой кода.

По умолчанию инференс НЕ перезапускается: `responses.jsonl` и ручная разметка `manual-M1` уже закоммичены в `results/`. Ячейки ниже сразу переходят к прогону LLM-судьи и сравнению с ней — аудио для этого не нужно (судья читает только текстовый транскрипт из манифеста).

In [ ]:
# ============================================================================
# PARAMETERS -- единственное место в этом ноутбуке, которое нужно редактировать
# руками. Дальше по коду только ЧИТАЕТ эти переменные -- ни одна ячейка ниже не
# требует ручного комментирования, чтобы Run All прошёл безопасно: каждая
# дорогая/опциональная операция управляется своим флагом отсюда (default =
# самый дешёвый/безопасный вариант).
# ============================================================================

# --- Репозиторий ---
# False (default): git clone из GitHub (нужен read-доступ к репо).
# True: доступа нет -- вместо клонирования распаковываем вручную загруженный zip (REPO_ZIP_PATH).
USE_LOCAL_ZIP = False
REPO_ZIP_PATH = "/kaggle/input/slm-audio-evidence/slm-audio-evidence.zip"
# 2026-07-22: это НАСТОЯЩИЙ актуальный форк -- см. docs/decisions.md (старый URL указывал на
# форк, отставший на 23 коммита; Kaggle тихо гонял старый src/*.py весь прошлый прогон).
REPO_URL = "https://github.com/PolinaSh-main/slm-audio-evidence-judge-fork.git"
REPO_BRANCH = "m3/llm-judge"
LOCAL_DIR = "slm-audio-evidence"  # явно -- иначе git clone называет папку по имени репо
# ("slm-audio-evidence-judge-fork"), а не REPO_URL, ломая %cd ниже.

# --- Инференс (сырое аудио -> responses.jsonl) ---
# True -- только чтобы пересчитать responses.jsonl с нуля (новые аудио/модели/промпты).
# False (default): пропустить аудио + инференс целиком -- responses.jsonl и ручная разметка
# manual-M1 уже закоммичены в results/.
RUN_INFERENCE = False
AUDIO_ZIP_PATH = "/kaggle/input/pilot-audio/pilot_audio.zip"  # читается только если RUN_INFERENCE

# --- Судья: общее для ВСЕХ кандидатов лестницы ниже ---
RUNS = [
    "qwen2audio_plain_20260712",
    "qwen2audio_s1_idk_20260712",
    "cascade_plain_20260712",
    "cascade_s1_idk_20260712",
]
# "dev_subset" (быстро, ~23 отобранных вопроса, намеренно смещённых в сторону прошлых
# несогласий с manual-M1 -- ОЖИДАЕМО ниже % чем на "full", это не регресс) / "full" (100%,
# 93 вопроса -- то число, что идёт в PR/отчёт).
JUDGE_MODE = "dev_subset"
HF_CACHE_DIR = "/root/hf_cache"  # НЕ /kaggle/working -- см. markdown ниже про квоту Output'а

# Тумблер: обычная рубрика judge_v1.txt vs G-Eval-style промпт с явными evaluation steps
# (geval_v1.txt). См. docs/decisions.md 2026-07-23 -- единственная реальная проблема thinking-
# режима (снисходительность на абстрактных/обобщающих gold-ответах) -- ровно то, во что бьёт
# 5-й шаг geval_v1.txt, БЕЗ необходимости в reasoning-модели вообще. Переключи, прогони ячейки
# кандидатов, сравни с прогоном на judge_v1.txt тех же моделей.
# ВАЖНО (docs/decisions.md 2026-07-23): geval_v1.txt пишет свои 5 шагов ВИДИМЫМ текстом (не в
# скрытом <think>-блоке) -- на no-think кандидате первый же реальный прогон обрезался на 64
# токенах (дефолт под однословный judge_v1.txt) прямо на середине Step 1 у 11/93 вопросов,
# сплошь UNPARSEABLE. Поэтому ячейки кандидатов ниже сами поднимают max_new_tokens при
# включённом тумблере (см. CANDIDATE_MAX_NEW_TOKENS в ячейке run_candidate) -- менять здесь
# ничего дополнительно не нужно.
USE_GEVAL_PROMPT = False

# Переиспользование весов между сессиями, по модели (см. markdown ниже): model_id -> путь
# монтирования прикреплённого Kaggle Dataset, если сохраняла (SAVE_FLAT_CACHE ниже). Модели,
# которых здесь нет (или чей путь не примонтирован в этой сессии), просто качаются с Hub заново.
FLAT_CACHE_DIRS: dict = {
    "Qwen/Qwen3-8B": "/kaggle/input/datasets/pasheviakova/qwen3-8b-flat-cache",
    "mistralai/Ministral-3-8B-Instruct-2512-BF16":
        "/kaggle/input/datasets/pasheviakova/llm-ministral-3-8b-instruct-2512-v1-flat-cache",
    "mistralai/Ministral-3-8B-Reasoning-2512":
        "/kaggle/input/datasets/pasheviakova/llm-ministral-3-8b-reasoning-251-v2-flat-cache",
    "Qwen/Qwen3.5-9B":
        "/kaggle/input/datasets/pasheviakova/llm-qwen3-5-9b-v1-flat-cache",
    "google/gemma-4-12b-it":
        "/kaggle/input/datasets/pasheviakova/llm-gemma-4-12b-it-v1-flat-cache",
}

# Выключено по умолчанию: примерно удваивает дисковый I/O на кандидата (сохранить плоскую копию
# + загрузить, поверх самого скачивания) и добавляет реальное время (загрузка десятков ГБ на
# модель) -- включай только для кандидата, который реально хочешь переиспользовать между
# сессиями без повторного скачивания.
SAVE_FLAT_CACHE = False
KAGGLE_USERNAME = "pasheviakova"  # поправь, если логин другой

# Точечный эксперимент (см. markdown ближе к концу ноутбука) -- по умолчанию выключен, чтобы
# Run All не платил лишней загрузкой модели ради него. Строит СВОЙ отдельный локальный судья.
RUN_ADHOC_APOLLO13_V5 = False

print(
    f"Parameters: RUN_INFERENCE={RUN_INFERENCE}, JUDGE_MODE={JUDGE_MODE!r}, "
    f"USE_GEVAL_PROMPT={USE_GEVAL_PROMPT}, SAVE_FLAT_CACHE={SAVE_FLAT_CACHE}, "
    f"RUN_ADHOC_APOLLO13_V5={RUN_ADHOC_APOLLO13_V5}"
)

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

import os

if USE_LOCAL_ZIP:
    sh(f"mkdir -p {LOCAL_DIR} && unzip -q -o {REPO_ZIP_PATH} -d {LOCAL_DIR}")
elif os.path.isdir(LOCAL_DIR):
    # 2026-07-22: a plain `git clone` here fails outright if this cell (or an earlier session in
    # the same warm kernel/Working dir) already checked out LOCAL_DIR once -- "destination path
    # ... already exists". Update in place instead of cloning fresh. `git remote set-url` first
    # (not just `git pull`) so this is also correct if LOCAL_DIR was cloned from a since-changed
    # REPO_URL (e.g. the stale-fork -> real-fork switch earlier this project) -- always ends up
    # tracking whatever REPO_URL/REPO_BRANCH say right now, regardless of prior state. Hard
    # reset, not merge -- this notebook only ever reads the repo, never commits into this clone,
    # so there is nothing local here worth preserving.
    sh(
        f"cd {LOCAL_DIR} && git remote set-url origin {REPO_URL} && "
        f"git fetch origin {REPO_BRANCH} && git checkout {REPO_BRANCH} && "
        f"git reset --hard origin/{REPO_BRANCH}"
    )
else:
    sh(f"git clone --branch {REPO_BRANCH} {REPO_URL} {LOCAL_DIR}")
%cd slm-audio-evidence

**Если меняешь пин `transformers` в `requirements.txt` (или гонишь `pip install -U ...` вручную) в уже тёплом kernel'е — этого недостаточно.** `pip install` обновляет пакет только на диске; если `transformers` уже был импортирован в этом процессе раньше (например, предыдущий кандидат из лестницы ниже уже успешно отработал), Python держит СТАРЫЙ модуль в `sys.modules`, и апгрейд молча не подхватывается — `pip show transformers` покажет новую версию, а реальные вызовы (`AutoConfig.from_pretrained` и т.п.) всё равно будут падать со старым поведением. См. `docs/decisions.md` 2026-07-23 (Gemma-4/Qwen3.5/Qwen3.6, `KeyError: 'qwen3_5'` несмотря на честные `5.14.1` в `pip show`) — та же ловушка, что уже описана ниже для `HF_HOME`. Единственный надёжный способ подхватить новую версию — **`Run -> Restart session`**, затем заново сверху (clone -> pip install -> GPU-check -> кандидат), а не просто повторный запуск ячеек.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible -- Settings (right panel) -> Accelerator, pick GPU T4 x2, "
        "then Restart session and rerun from the clone cell."
    )

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute capability {capability[0]}.{capability[1]})")
if capability < (7, 0):
    raise RuntimeError(
        f"{name} (compute capability {capability[0]}.{capability[1]}) is too old for the "
        "preinstalled PyTorch build (needs >= 7.0, e.g. T4/V100/A100). Seen in practice: "
        "P100 (Pascal, 6.0) fails here. Settings (right panel) -> Accelerator -> switch to "
        "GPU T4 x2, then Restart session and rerun from the clone cell."
    )

In [ ]:
if RUN_INFERENCE:
    sh(f'unzip -q -o {AUDIO_ZIP_PATH} -d .')
    sh('ls data/audio/spoken_squad_test | head -3')
    import json
    rows = [json.loads(l) for l in open('data/manifests/pilot.jsonl', encoding='utf-8')]
    miss = [r['id'] for r in rows if not os.path.exists(r['audio_path'])]
    print(len(rows), 'items,', len(miss), 'missing audio')
    print(miss[:5])
else:
    print('RUN_INFERENCE=False -- skipping audio upload/check (responses.jsonl already in results/).')

In [ ]:
# Run 1 (the headline): Qwen2-Audio, plain prompt. First run also downloads the weights (~15-30 min).
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 2: Qwen2-Audio, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# IMPORTANT before runs 3-4: free VRAM from Qwen2-Audio -- restart the session/kernel
# (Kaggle: top menu -> Run -> Restart session, or the restart icon), then re-run the clone,
# pip install, GPU-check and Parameters cells above -- then continue here.
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 4: cascade, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Zip everything produced so far (run after EACH finished run -- do not wait for all four)
if RUN_INFERENCE:
    sh("zip -q -r results_runs.zip results -x '*.gitkeep'")
    print('results_runs.zip written -- grab it from the notebook Output tab '
          '(Kaggle persists everything under /kaggle/working) once the session ends, or via the file browser now.')
else:
    print('RUN_INFERENCE=False -- nothing new to zip.')

## LLM-судья vs ручная разметка manual-M1

`results/<run_id>/responses.jsonl` уже есть (см. выше). Категория B (label=answer) в них размечена людьми вручную (`results/<run_id>/responses_judged.jsonl`, `judge: "manual-M1"`, committed — см. docs/decisions.md) — это уже готовая истина, повторную слепую разметку делать не нужно.

**Единая лестница кандидатов, один прогон.** Раньше здесь было ДВЕ параллельные системы — отдельный "замороженный" прогон Qwen3-8B и отдельная "лестница кандидатов" для остальных моделей, каждая со своим именованием папок, своим скриптом сравнения и общий их набор параметров надо было держать в голове отдельно. Теперь Qwen3-8B — это просто ПЕРВЫЙ кандидат в той же самой лестнице, с тем же `run_candidate()`, тем же именованием (`llm_audit_<judge.name>`) и тем же скриптом сравнения (`audit_multi_judge.py`), что и все остальные модели. Значит: `USE_GEVAL_PROMPT` из Parameters действует на Qwen3-8B ровно так же, как на любую другую модель — **больше нет отдельного кода, который его игнорирует.**

Судья пишет в **отдельную** папку `results/<run_id>/llm_audit_<judge.name>/` на КАЖДОГО кандидата — уже размеченный `responses_judged.jsonl` (ручная разметка manual-M1) не трогается никем.

**ЗАМОРОЖЕННЫЙ рабочий конфиг — первый кандидат ниже (`Qwen/Qwen3-8B`, `judge_v1.txt`, no-think): 94% (87/93).** Вся серия экспериментов над промптом и режимом (`judge_v2.txt`, thinking, `judge_v3.txt` ± thinking, `judge_v4.txt` без транскрипта) провалилась — ни один вариант не побил простой no-think `judge_v1.txt`. См. docs/decisions.md 2026-07-15/07-16. Меняя его конфиг в коде ниже, ты не "правишь баг", а ставишь новый эксперимент — не перезаписывай эту строку тихо.

**`judge_v4.txt` (v1 без строки TRANSCRIPT) — проверен на полном датасете, ОТКЛОНЁН.** Гипотеза была: раз транскрипт не упомянут в критерии вердикта дословно, может, он и не нужен — и это сэкономило бы токены на prefill. Оказалось не так: 90% (84/93) против 94% (87/93) — минус 4 пункта. Разбор по элементам: 1 случай починился, но 5 сломались, все на коротких легитимных ответах (например `sq-1122-B1`: "Secondary sources of European Union law are based on the treaties." — верно, но короче gold — без транскрипта в контексте судья резче наказывает такую краткость). Вывод: транскрипт не используется explicitly по тексту рубрики, но реально работает как стабилизирующий контекст — держим его в `judge_v1.txt`.

**Кэш весов между сессиями (docs/decisions.md 2026-07-22)**: сырой HF-кэш (`blobs`+`snapshots`-ссылки) как Kaggle Dataset НЕ работает надёжно — Kaggle плохо сохраняет ссылки внутри файлов при загрузке датасета, и `snapshots/` (без которой нельзя понять, где какой файл) может потеряться. Вместо этого — плоские именованные файлы (`model.save_pretrained`/`tokenizer.save_pretrained`), которые Dataset уже не испортит: выставь `SAVE_FLAT_CACHE = True` в Parameters, и КАЖДЫЙ `local`-кандидат ниже сам сохранится плоско и загрузится как Kaggle Dataset `<логин>/<имя-модели>-flat-cache` — реализовано один раз внутри `run_candidate()`, а не вручную под каждую модель.

Чтобы ПЕРЕИСПОЛЬЗОВАТЬ уже сохранённый плоский кэш в новой сессии — прикрепи датасет через **Add Data** и добавь его путь монтирования в `FLAT_CACHE_DIRS` (в Parameters) под соответствующим `model_id`. Путь монтирования на этом аккаунте на уровень глубже, чем можно подумать — **`/kaggle/input/datasets/<логин>/<slug>`**, не `/kaggle/input/<slug>` (сверься с `!ls -la /kaggle/input/datasets/*/`, если сомневаешься).

**Про `HF_HUB_OFFLINE` (пробовали 2026-07-22, откатили)**: форсировать оффлайн-режим после копирования кэша выглядело логично, но `from_pretrained` и без этого флага уже вёл себя правильно — короткая проверка ревизии на Hub (пара КБ, видна в логе как "Fetching N files" / "Download complete: 0.00/0.00" — это НЕ повторная закачка) и переиспользование локальных весов без скачивания. Форсированный оффлайн-режим требует, чтобы кэш был идеально самодостаточен без права на сетевую подстраховку — не стоит той хрупкости, которую даёт полный оффлайн. Флаг не используется.

In [ ]:
import gc
import json
import re
import time
from pathlib import Path

import torch
from huggingface_hub import scan_cache_dir

from src.judges import build_judge
from src.judges.dev_subset import DEV_SUBSET
from src.run_eval import run_evaluation

# Kaggle Secrets -> GEMINI_API_KEY, only if the "gemini" candidate below is actually used.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
except Exception:
    pass  # no Secret attached -- GeminiJudge raises a clear RuntimeError if gemini is selected without it

# NOT under /kaggle/working -- that's the notebook's persisted "Output" with a hard quota
# (~19 GB on this account); several candidates below are 15-55 GB each, so keeping all of them
# there at once would blow the quota on the first one or two. /root/hf_cache lives on the
# container's separate, much bigger local disk -- doesn't persist across sessions on its own,
# but that's fine: only results/ needs to survive, not the raw weights.
#
# IMPORTANT: HF_HOME is read by huggingface_hub ONCE, at its first import in this process --
# setting it here only works if this is the FIRST time in this kernel that
# transformers/huggingface_hub gets imported (i.e. right after Run -> Restart session).
# Re-running just this cell in an already-warm kernel silently keeps the old cache dir.
os.environ["HF_HOME"] = HF_CACHE_DIR

CANDIDATE_PROMPT_NAME = "geval_v1.txt" if USE_GEVAL_PROMPT else "judge_v1.txt"
# Only non-empty when NOT the default prompt -- keeps judge_v1.txt output dirs/manifest entries
# byte-identical to a run without this toggle, so a judge_v1.txt pass and a geval_v1.txt pass of
# the same model never collide/overwrite each other on disk.
_PROMPT_SUFFIX = "" if CANDIDATE_PROMPT_NAME == "judge_v1.txt" else "_" + CANDIDATE_PROMPT_NAME.replace(".txt", "")

# 2026-07-23 (docs/decisions.md): geval_v1.txt asks the model to write its 5 evaluation steps as
# VISIBLE output (this is separate from enable_thinking's hidden <think> block -- a no-think
# candidate writes the steps directly into its answer), but max_new_tokens still defaulted to 64
# (LocalHFJudge's no-think default, tuned for judge_v1.txt's bare one-word reply). First real run
# (Qwen3-8B, no-think) truncated 11/93 items mid-Step-1 at ~64 tokens, all UNPARSEABLE, before
# ever reaching a verdict word -- confirmed from the raw outputs, not guessed.
#
# 4096, not a smaller headroom value: this is a CEILING, not a forced length -- _StopOnVerdict
# (src/judges/local_hf.py) only cuts generation short once it sees one of _THINK_CLOSE_MARKERS,
# and geval_v1.txt never emits one (no thinking involved here), so early stopping never fires for
# these candidates either way; the actual length is bounded by the model's own EOS token, not by
# this number. 4096 just guarantees every step + the verdict always fits regardless of how
# verbose a given reply gets, at no extra cost for the (expected) common case where the model
# finishes well under it. Candidates that already pass their own explicit max_new_tokens (e.g.
# Ministral-Reasoning's 4096) are unaffected -- takes precedence in that cell's own kwargs dict.
GEVAL_MAX_NEW_TOKENS = 4096
CANDIDATE_MAX_NEW_TOKENS = GEVAL_MAX_NEW_TOKENS if USE_GEVAL_PROMPT else None

MANIFEST_PATH = "results/judge_audit_multi/run_manifest.json"


def _resolve_model_id(hub_id: str) -> dict:
    """kwargs for build_judge: {"model_id": ...} normally, or {"model_id": <local flat-cache
    dir>, "display_name": hub_id} when a flat cache is attached for hub_id this session
    (FLAT_CACHE_DIRS in Parameters). display_name matters -- LocalHFJudge derives judge.name
    (the judge_cache.jsonl cache key / output-dir component) from it, not from model_id, so a
    flat-cache run and a fresh-Hub-download run of the exact same model always share the same
    identity instead of silently forking into "llm-qwen3-8b-v1" vs
    "llm-qwen3-8b-flat-cache-v1" (caught in practice, docs/decisions.md 2026-07-23).

    The presence of "display_name" in the returned dict is also used by run_candidate() below as
    the signal that this candidate was loaded from an EXISTING flat-cache mount rather than fresh
    from the Hub -- see the SAVE_FLAT_CACHE skip-if-already-cached note there.
    """
    flat_dir = FLAT_CACHE_DIRS.get(hub_id)
    if flat_dir and os.path.isdir(flat_dir):
        sh(f"du -sh {flat_dir}")  # sanity check -- should be several GB, not KB/MB
        print(f"Using flat cache for {hub_id}: {flat_dir}")
        return {"model_id": flat_dir, "display_name": hub_id}
    return {"model_id": hub_id}


def _slugify(name: str, max_len: int = 35) -> str:
    """Kaggle dataset handles: lowercase letters/digits/hyphens only, and there's a length cap
    on the slug (not 100% confirmed exact number -- 35 is a conservative guess so
    "<slug>-flat-cache" stays comfortably short). Truncating naively from the right can eat the
    "-v1"/"-v2" suffix on long names (e.g. Ministral's) and make two different candidates collide
    on the same dataset handle -- so the suffix is carved out first and always preserved,
    only the model-name part in the middle gets shortened.
    """
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    m = re.search(r"-(v\d+)$", slug)
    suffix = f"-{m.group(1)}" if m else ""
    base = slug[: -len(suffix)] if suffix else slug
    return base[: max_len - len(suffix)].rstrip("-") + suffix


def save_flat_and_upload(cand_judge, max_attempts: int = 3) -> None:
    """save_pretrained + kagglehub.dataset_upload -- see markdown above (raw HF cache loses its
    snapshots/ symlinks when packaged as a Kaggle Dataset; flat named files don't have that
    problem). Writes to /root, not /kaggle/working -- some candidates here are tens of GB,
    bigger than the ~19 GB Output quota entirely.

    Called by run_candidate() right after the model loads, BEFORE the eval loop (2026-07-23,
    docs/decisions.md) -- see the note there for why.

    2026-07-23 (docs/decisions.md): flat_dir is now ALWAYS deleted after the upload attempt(s)
    (success or final failure) -- it previously lingered on disk forever, meaning every
    SAVE_FLAT_CACHE candidate left a second full copy of its own weights sitting on the
    container's local disk (on top of whatever free_disk_cache_for() frees from HF_HOME) for the
    rest of the session. Caught in practice: Kaggle's "running out of disk space" warning
    mid-ladder, in a session that had already run one SAVE_FLAT_CACHE candidate before this one.
    It's only a local staging copy for the upload -- once the upload call returns (either way),
    nothing needs it anymore.

    max_attempts=3: `kagglehub.dataset_upload`'s final `create_dataset` RPC has been observed
    returning an empty/non-JSON body (raises deep inside the SDK as a plain JSONDecodeError, not
    a structured kagglehub error) AFTER every file already uploaded successfully -- looks like a
    one-off Kaggle API hiccup on the finalize step, not a data/parameter problem (the actual
    multi-GB file transfers all completed fine). Worth a couple of retries with backoff before
    giving up -- re-running the whole candidate (reload model + re-eval) just to retry this one
    RPC would be far more wasteful than retrying the RPC itself.
    """
    import shutil
    import time as _time

    import kagglehub

    slug = _slugify(cand_judge.name)
    flat_dir = f"/root/{slug}_flat"
    handle = f"{KAGGLE_USERNAME}/{slug}-flat-cache"

    cand_judge.model.save_pretrained(flat_dir)
    cand_judge.tokenizer.save_pretrained(flat_dir)
    sh(f"du -sh {flat_dir}")  # sanity check -- should be several GB, not KB/MB

    try:
        last_exc: Exception | None = None
        for attempt in range(1, max_attempts + 1):
            try:
                kagglehub.dataset_upload(handle, flat_dir)
                last_exc = None
                break
            except Exception as exc:
                last_exc = exc
                if attempt < max_attempts:
                    wait_s = 15 * attempt
                    print(f"  upload attempt {attempt}/{max_attempts} failed ({exc!r}) -- "
                          f"retrying in {wait_s}s...")
                    _time.sleep(wait_s)
        if last_exc is not None:
            raise last_exc
        # Confirmed 2026-07-22 on this account: the mount point is one level deeper than the
        # dataset slug alone -- "/kaggle/input/datasets/<username>/<slug>/", not
        # "/kaggle/input/<slug>/".
        expected_mount = f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{slug}-flat-cache"
        print(f"  uploaded {handle} from {flat_dir} -- attach via Add Data next session, then add "
              f'"{expected_mount}" to FLAT_CACHE_DIRS in Parameters '
              f"(verify with !ls -la /kaggle/input/datasets/*/).")
    finally:
        shutil.rmtree(flat_dir, ignore_errors=True)


def free_disk_cache_for(model_id: str) -> None:
    """Delete this model's downloaded weights from HF_HOME once we're done with it. Even off
    /kaggle/working, the container's local disk is finite -- several candidates are tens of GB
    each (Qwen3.6-27B alone is ~54 GB bf16), holding all of them on disk at once would still run
    out of room. Only the raw weight cache is freed -- results/ is untouched.

    2026-07-23: a candidate loaded entirely from a flat-cache dir (FLAT_CACHE_DIRS) never
    touches the Hub cache at all -- HF_HOME may not even exist on disk yet in that case (nothing
    else downloaded anything this session either), and scan_cache_dir() raises CacheNotFound
    for a missing directory rather than treating it as "nothing cached". Caught in practice on
    Qwen3-8B's flat-cache run. Nothing to free either way, so just skip.
    """
    if not os.path.isdir(os.environ["HF_HOME"]):
        return
    cache_info = scan_cache_dir(os.environ["HF_HOME"])
    revisions = [rev.commit_hash for repo in cache_info.repos if repo.repo_id == model_id for rev in repo.revisions]
    if revisions:
        cache_info.delete_revisions(*revisions).execute()
        print(f"  freed disk cache for {model_id}")


def _append_manifest_entry(entry: dict) -> None:
    """Each candidate cell calls this once -- results/judge_audit_multi/run_manifest.json is
    always up to date with whatever's actually been run so far, survives a crash in a later
    cell, and a rerun of one model's cell just replaces its own entry (matched by judge_name)
    instead of duplicating it.
    """
    path = Path(MANIFEST_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    entries = json.loads(path.read_text(encoding="utf-8")) if path.exists() else []
    entries = [e for e in entries if e["judge_name"] != entry["judge_name"]]
    entries.append(entry)
    path.write_text(json.dumps(entries, ensure_ascii=False, indent=2), encoding="utf-8")


def run_candidate(backend: str, kwargs: dict, thinking: bool, free_disk_cache: bool = True) -> None:
    """Run one judge candidate across all RUNS and record it in the shared manifest. Pass
    free_disk_cache=False if the very next cell you're about to run reuses the same model_id
    (e.g. Qwen3.5-9B no-think then thinking) -- skips a pointless redownload.

    Wrapped in try/finally: GPU cleanup must run even if a candidate fails partway (e.g. a
    checkpoint key mismatch, or any run_evaluation error) -- otherwise its model stays resident
    in VRAM and the NEXT candidate fails with bitsandbytes' "dispatched on the CPU or the disk"
    refusal even though it would easily have fit on its own (docs/decisions.md 2026-07-22).
    """
    cand_judge = None
    try:
        cand_judge = build_judge(backend, prompt_name=CANDIDATE_PROMPT_NAME, **kwargs)
        display_name = cand_judge.name + _PROMPT_SUFFIX
        print(f"=== {display_name} ===")

        if backend == "local" and SAVE_FLAT_CACHE and "display_name" not in kwargs:
            # 2026-07-23 (docs/decisions.md): moved BEFORE the eval loop below -- it used to run
            # only after all of RUNS finished, so a crash/timeout partway through a long eval
            # (dev_subset alone is tens of minutes per candidate; full is much more) meant the
            # already-downloaded weights never got saved/uploaded, even though nothing about the
            # eval itself was needed to do that. Saving right after the model loads means the
            # cache survives even if this candidate's eval never finishes this session.
            #
            # "display_name" in kwargs means _resolve_model_id() already redirected this candidate
            # to an EXISTING flat-cache mount (FLAT_CACHE_DIRS) -- nothing new to save, so
            # re-uploading the exact same weights back to the exact same Kaggle Dataset would be
            # pure waste. Only candidates actually loaded fresh from the Hub go through this path.
            #
            # save_flat_and_upload() already retries the flaky Kaggle upload a few times
            # internally -- this outer try/except is the last resort if it still fails after all
            # retries. This is a cache-reuse OPTIMIZATION, not the actual research output, so a
            # failure here must not block the eval that follows.
            try:
                save_flat_and_upload(cand_judge)
            except Exception as exc:
                print(f"  WARNING: save_flat_and_upload failed ({exc!r}) -- continuing to the "
                      "eval below anyway; only the cross-session flat-cache upload is missing. "
                      "See docs/decisions.md 2026-07-23.")

        t0 = time.time()
        for run_id in RUNS:
            subset_ids = set(DEV_SUBSET.get(run_id, [])) if JUDGE_MODE == "dev_subset" else None
            run_evaluation(
                manifest_path="data/manifests/pilot.jsonl",
                responses_path=f"results/{run_id}/responses.jsonl",
                out_dir=f"results/{run_id}/llm_audit_{display_name}",  # own dir per candidate+prompt
                judge=cand_judge,
                subset_ids=subset_ids,
            )
        elapsed = time.time() - t0
        print(f"=== {display_name} done in {elapsed:.0f}s ===")
        _append_manifest_entry({
            "judge_name": display_name,
            "model_id": kwargs.get("model_id", backend),
            "backend": backend,
            "thinking": thinking,
            "prompt_name": CANDIDATE_PROMPT_NAME,
            "seconds": elapsed,
        })
    finally:
        # Runs even if build_judge()/run_evaluation() raised above -- cand_judge may still be
        # None (failed before a model object even existed), that's fine, nothing to free then.
        if cand_judge is not None:
            if backend == "local" and free_disk_cache:
                free_disk_cache_for(kwargs["model_id"])
            del cand_judge
        gc.collect()  # unconditional -- also helps release VRAM fragments from a load that
        torch.cuda.empty_cache()  # raised partway through, not just a clean cand_judge del.


print(f"Setup done -- USE_GEVAL_PROMPT={USE_GEVAL_PROMPT} ({CANDIDATE_PROMPT_NAME}). "
      "Run any of the per-model cells below, in any order.")

## Лестница судей-кандидатов

Каждая модель — в своей ОТДЕЛЬНОЙ ячейке (см. `run_candidate()` выше): можно запускать/перезапускать любую независимо от остальных, в любом порядке, пропускать те, что пока не нужны, и повторить один упавший прогон, не трогая уже готовые. Каждый кандидат пишет вердикты в СВОЮ папку `results/<run_id>/llm_audit_<judge.name>/` — `responses_judged.jsonl` (ручная разметка manual-M1) не трогается ни при каком кандидате.

**`model_id` проверены напрямую на huggingface.co 2026-07-21** (не плейсхолдеры):
- `mistralai/Ministral-3-8B-Instruct-2512-BF16` — не "Mistral-3", а **Ministral-3** (линейка Mistral для edge/small моделей); Apache 2.0. У Instruct-варианта thinking-переключателя НЕТ вообще. Reasoning — **отдельный чекпоинт** `mistralai/Ministral-3-8B-Reasoning-2512`, тоже без переключателя, но всегда рассуждает (как DeepSeek-R1-Distill — дешёвого no-think режима для него не существует).
- `Qwen/Qwen3.5-9B` и `Qwen/Qwen3.6-27B` (не `-Instruct`, это сами чат-модели, как и наш `Qwen3-8B`) — оба Apache 2.0, оба подтверждают `enable_thinking` в `apply_chat_template` (тот же интерфейс, что уже реализован в `LocalHFJudge`).

**Важная находка при проверке**: `Ministral-3-8B`, `Qwen3.5-9B` и `Qwen3.6-27B` — все три оказались vision-language моделями (multimodal, с vision-энкодером), не чисто текстовыми. `LocalHFJudge` (`src/judges/local_hf.py`) при необходимости сама подхватывает `AutoModelForImageTextToText` вместо `AutoModelForCausalLM` (см. docstring в коде) — если всё же увидишь ошибку класса на новой модели, это реальный пробел в `LocalHFJudge`, а не опечатка в `model_id`.

Для Gemma-4 (`google/gemma-4-12b-it`) отдельно проверь в её `config.json`/model card, действительно ли `enable_thinking` переключается в рантайме, или reasoning встроен всегда (источники противоречат друг другу) — если встроен всегда, `enable_thinking=False` ничего не изменит и это НЕ баг.

Gemini не грузит веса — вместо GPU нужен `GEMINI_API_KEY`. Положи его в Kaggle Secrets (правая панель → Add-ons → Secrets). Модель зафиксирована на `gemini-2.5-flash-lite` (docs/decisions.md 2026-07-22) — единственная free-tier модель Gemini, чей дневной лимит (1000 запросов) реально покрывает прогон на весь пилот; `gemini-2.5-flash` упирается в квоту (250/день) на середине одного прогона. Не подменяем модель на лету при исчерпании квоты (осознанно отклонённая идея — вердикты от разных judge-моделей внутри одного датасета несравнимы), только пауза между вызовами + честное ожидание.

**Qwen3-8B — ЗАМОРОЖЕННЫЙ базовый конфиг, 94% (87/93).** Первый кандидат, не эксперимент — см. markdown про `judge_v1.txt`/`judge_v4.txt` выше. Меняя `model_id`/`enable_thinking`/промпт здесь, ты создаёшь новый эксперимент поверх этой строки, не тихо переписываешь baseline.

In [ ]:
run_candidate("local", {**_resolve_model_id("Qwen/Qwen3-8B"), "enable_thinking": False, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=False)

**Gemini 2.5 Flash-Lite** — не грузит GPU, нужен `GEMINI_API_KEY` в Kaggle Secrets (см. markdown выше). Проактивно троттлится под free-tier RPM внутри `GeminiJudge` — обычно не должно требовать ретраев вовсе.

In [ ]:
run_candidate("gemini", {"model_id": "gemini-2.5-flash-lite"}, thinking=False)

**Ministral-3-8B Instruct** — VLM checkpoint (see markdown above), no thinking toggle at all.

In [ ]:
run_candidate("local", {**_resolve_model_id("mistralai/Ministral-3-8B-Instruct-2512-BF16"), "enable_thinking": False, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=False)

**Ministral-3-8B Reasoning** — separate always-on checkpoint (no toggle, always reasons, like DeepSeek-R1-Distill). Also a VLM checkpoint -- same class-mismatch caveat as above.

`max_new_tokens=4096` (2026-07-22, up from the 512 default): Mistral's own reasoning-close marker is `[/THINK]`, not Qwen3's `</think>` -- `_THINK_CLOSE_MARKERS` in `src/judges/base.py` now recognizes both, but several items still ran out of budget mid-reasoning before reaching either marker at all (verified from real raw_output text, not guessed). Considered budget forcing (force-inject the closing marker + a short follow-up generate() once the cap is hit) but rejected it -- it fights the project's own "never guess an ambiguous verdict" principle (parse_verdict's UNPARSEABLE contract). A generous fixed cap is simpler and any item still UNPARSEABLE past 4096 is a genuine signal, not an artifact of too tight a budget.

**2026-07-23 finding (docs/decisions.md): thinking here ties no-think's accuracy (92% vs 92%) but costs 9640s and ~20% UNPARSEABLE coverage loss** -- its one real failure mode is leniency on abstract/summary gold answers, which `geval_v1.txt` (the `USE_GEVAL_PROMPT` toggle in Parameters) targets directly without needing a reasoning model at all.

In [ ]:
run_candidate("local", {**_resolve_model_id("mistralai/Ministral-3-8B-Reasoning-2512"), "enable_thinking": True, "max_new_tokens": 4096}, thinking=True)

**Gemma-4-12B-it** — check its `config.json`/model card for whether `enable_thinking` actually toggles anything at runtime, or reasoning is always-on; if always-on, `enable_thinking=False` doing nothing is a model property, not a bug here.

In [ ]:
run_candidate("local", {**_resolve_model_id("google/gemma-4-12b-it"), "enable_thinking": False, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=False)

**Qwen3.5-9B, no-think then thinking** — same `model_id` in both of the next two cells, so the first one passes `free_disk_cache=False` to skip re-downloading for the second. Also a VLM checkpoint (see markdown above) -- watch for the same class-mismatch caveat.

In [ ]:
run_candidate("local", {**_resolve_model_id("Qwen/Qwen3.5-9B"), "enable_thinking": False, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=False, free_disk_cache=False)

In [ ]:
run_candidate("local", {**_resolve_model_id("Qwen/Qwen3.5-9B"), "enable_thinking": True, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=True)

**Qwen3.6-27B** — largest candidate (~54 GB bf16); also a VLM checkpoint, same class-mismatch caveat. Consider testing this one in isolation first given its size before running the rest of the ladder in the same session.

**2026-07-23 (docs/decisions.md): `load_in_4bit=True` instead of the default 8-bit.** 8-bit (~27 GB weights alone) left almost no headroom for KV-cache/activations on 2xT4 (32 GB combined) and failed with `ValueError: Some modules are dispatched on the CPU or the disk` -- accelerate's `device_map="auto"` tried to offload the overflow, which bitsandbytes' 8-bit path refuses outright. 4-bit halves the footprint (~13-14 GB) instead of fighting the offload path. Before assuming it's purely a size problem, first rule out leftover VRAM from a previous candidate (`!nvidia-smi --query-gpu=memory.used,memory.total --format=csv` -- should read close to 0 used before this cell runs; if not, `Run -> Restart session`).

In [ ]:
run_candidate("local", {**_resolve_model_id("Qwen/Qwen3.6-27B"), "enable_thinking": False, "load_in_4bit": True, "max_new_tokens": CANDIDATE_MAX_NEW_TOKENS}, thinking=False)

In [ ]:
# Optional: see which candidates have actually run so far (useful after running only a subset
# of the cells above) before generating the comparison table below.
print(Path(MANIFEST_PATH).read_text(encoding="utf-8") if Path(MANIFEST_PATH).exists()
      else f"{MANIFEST_PATH} does not exist yet -- run at least one candidate cell above first.")

## Сравнение и упаковка результатов

ОДНА команда собирает ОДНУ сравнительную таблицу по всем прогнанным кандидатам (agreement vs manual-M1, сортировка по убыванию, включая 80%-гейт ROLE_M3) + разложенные провалы каждой модели +, если был thinking, полные reasoning-трейсы по каждому вопросу — в отдельные папки, не в общий отчёт.

In [ ]:
sh(
    'python scripts/audit_multi_judge.py '
    '--manifest results/judge_audit_multi/run_manifest.json '
    '--judged "results/*/responses_judged.jsonl" '
    '--data-manifest data/manifests/pilot.jsonl '
    '--out results/judge_audit_multi'
)  # prints the comparison table itself -- no need to re-read/re-print it here

In [ ]:
# Собрать в zip: сравнительную таблицу, провалы каждой модели, и (отдельно) reasoning-трейсы
!zip -q -r judge_audit_multi.zip results/judge_audit_multi results/*/llm_audit_llm-* -x '*.gitkeep'
print('judge_audit_multi.zip written -- grab it from the notebook Output tab '
      '(Kaggle persists everything under /kaggle/working).')

### Точечный эксперимент: `judge_v5.txt` (без транскрипта) + thinking, только Apollo 13

Не часть основного цикла выше — маленькая ad-hoc проверка одной гипотезы на одном вопросе (`sq-5207-B2`), чтобы не платить за прогон всего `dev_subset` ради неё. Управляется `RUN_ADHOC_APOLLO13_V5` в Parameters (по умолчанию `False` — ничего не делает и не грузит модель). Строит СВОЮ отдельную копию `Qwen/Qwen3-8B` (а не переиспользует модель из кандидатской ячейки выше, как раньше) — раз каждая ячейка выше сама освобождает VRAM в `finally`, к моменту, когда доходишь сюда, гарантированно загруженной модели уже может не быть.

In [ ]:
# Ad-hoc, не часть основной лестницы: judge_v5.txt (judge_v3.txt без строки TRANSCRIPT) +
# thinking, только на sq-5207-B2 (Apollo 13).
if RUN_ADHOC_APOLLO13_V5:
    adhoc_judge = build_judge(
        "local", prompt_name="judge_v5.txt", enable_thinking=True, **_resolve_model_id("Qwen/Qwen3-8B"),
    )
    try:
        for run_id in RUNS:
            run_evaluation(
                manifest_path="data/manifests/pilot.jsonl",
                responses_path=f"results/{run_id}/responses.jsonl",
                out_dir=f"results/{run_id}/llm_audit_v5_adhoc_apollo13",
                judge=adhoc_judge,
                subset_ids={"sq-5207-B2"},  # only this one -- most runs will just skip it (not judge-routed)
            )
    finally:
        del adhoc_judge
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("RUN_ADHOC_APOLLO13_V5=False -- skipping (one-off judge_v5+thinking spot check, see markdown above).")